# Stage 5: Recommendation Scenario Simulation

## Goal

This notebook tests whether CLIP demographic-occupation association patterns can propagate into a simplified retrieval or recommendation setting. It uses embeddings and metadata produced by the main FairFace CLIP association notebook and does not recompute CLIP embeddings.


## Inputs

Required outputs from `src/notebooks/04_fairface_facet_broad_occupation_bias.ipynb`:

- `outputs/fairface_facet_broad/embeddings/image_embeddings.npy`
- `outputs/fairface_facet_broad/embeddings/text_embeddings.npy`
- `outputs/fairface_facet_broad/tables/sampled_fairface_labels.csv`
- `outputs/fairface_facet_broad/tables/occupation_prompts.csv`

If these files are missing, run the main notebook first.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from clip_bias.recommendation import (
    aggregate_topk_distribution,
    bootstrap_recommendation_disparity,
    compute_recommendation_disparity,
    normalize_rows,
    plot_recommendation_disparity_bar,
    plot_recommendation_sensitivity,
    plot_topk_distribution_heatmap,
    recommend_occupations_for_users,
    simulate_synthetic_users,
)

try:
    display
except NameError:
    display = print

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)

MAIN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "fairface_facet_broad"
MAIN_TABLE_DIR = MAIN_OUTPUT_DIR / "tables"
MAIN_EMBEDDING_DIR = MAIN_OUTPUT_DIR / "embeddings"
RECOMMENDATION_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "recommendation"
RECOMMENDATION_TABLE_DIR = RECOMMENDATION_OUTPUT_DIR / "tables"
RECOMMENDATION_FIGURE_DIR = RECOMMENDATION_OUTPUT_DIR / "figures"
RECOMMENDATION_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RECOMMENDATION_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

required_paths = {
    "sampled_labels": MAIN_TABLE_DIR / "sampled_fairface_labels.csv",
    "prompt_df": MAIN_TABLE_DIR / "occupation_prompts.csv",
    "image_embeddings": MAIN_EMBEDDING_DIR / "image_embeddings.npy",
    "text_embeddings": MAIN_EMBEDDING_DIR / "text_embeddings.npy",
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Stage 5 requires outputs from the main FairFace CLIP association notebook. "
        "Run src/notebooks/04_fairface_facet_broad_occupation_bias.ipynb first. Missing:
"
        + "
".join(missing)
    )

print("Project root:", PROJECT_ROOT)
print("Main outputs:", MAIN_OUTPUT_DIR)
print("Recommendation outputs:", RECOMMENDATION_OUTPUT_DIR)


## Simulation Design

The simulation constructs synthetic user profiles from historical image embeddings sampled within one demographic group. Occupation prompt embeddings are treated as retrieval candidates. The notebook then compares top-k recommendation distributions across demographic groups.


In [ ]:
sampled_labels = pd.read_csv(required_paths["sampled_labels"])
prompt_df = pd.read_csv(required_paths["prompt_df"])
image_embeddings = np.load(required_paths["image_embeddings"])
text_embeddings = np.load(required_paths["text_embeddings"])

if len(sampled_labels) != image_embeddings.shape[0]:
    raise ValueError(
        f"sampled_labels rows ({len(sampled_labels)}) do not match image_embeddings rows ({image_embeddings.shape[0]})."
    )
if len(prompt_df) != text_embeddings.shape[0]:
    raise ValueError(
        f"prompt_df rows ({len(prompt_df)}) do not match text_embeddings rows ({text_embeddings.shape[0]})."
    )

sim_metadata = sampled_labels.copy()
if "race_gender" not in sim_metadata.columns:
    sim_metadata["race_gender"] = sim_metadata["race"].astype(str) + "_" + sim_metadata["gender"].astype(str)

sim_group_col = "race_gender"
sim_prompt_df = prompt_df.reset_index(drop=True).copy()
sim_prompt_embeddings = normalize_rows(text_embeddings)

sim_occupation_names = sim_prompt_df["occupation"].drop_duplicates().tolist()
sim_text_embeddings = []
for occupation in sim_occupation_names:
    occupation_indices = sim_prompt_df.index[sim_prompt_df["occupation"] == occupation].to_numpy()
    sim_text_embeddings.append(sim_prompt_embeddings[occupation_indices].mean(axis=0))
sim_text_embeddings = normalize_rows(np.vstack(sim_text_embeddings))

print("Images:", image_embeddings.shape)
print("Prompt embeddings:", text_embeddings.shape)
print("Occupation candidates:", len(sim_occupation_names))
display(sim_metadata[["race", "gender", "race_gender"]].head())


## Synthetic User Construction

Each synthetic user is represented by the average of `history_size` normalized CLIP image embeddings sampled from one demographic group. The averaged embedding is normalized again before scoring. This creates controlled group-level profiles for testing propagation, not real user profiles.


## Recommendation Scoring

For each synthetic user embedding, occupations are ranked by cosine similarity against occupation-level text embeddings. Because the main notebook encodes multiple prompt templates per occupation, this notebook averages prompt embeddings within each occupation to form one candidate embedding per occupation.


In [ ]:
user_meta, user_emb = simulate_synthetic_users(
    image_embeddings=image_embeddings,
    metadata=sim_metadata,
    group_col=sim_group_col,
    n_users_per_group=100,
    history_size=10,
    random_state=42,
)

recommendations = recommend_occupations_for_users(
    user_embeddings=user_emb,
    user_metadata=user_meta,
    text_embeddings=sim_text_embeddings,
    occupation_names=sim_occupation_names,
    top_k=5,
)

recommendations_path = RECOMMENDATION_TABLE_DIR / "synthetic_user_recommendations.csv"
recommendations.to_csv(recommendations_path, index=False)

display(recommendations.head())
print("Saved:", recommendations_path)


## Main Top-k Distribution Result


In [ ]:
topk_dist = aggregate_topk_distribution(
    recommendations,
    group_col="group",
    all_occupations=sim_occupation_names,
)
rec_disparity = compute_recommendation_disparity(topk_dist)

topk_dist_path = RECOMMENDATION_TABLE_DIR / "topk_occupation_distribution_by_group.csv"
rec_disparity_path = RECOMMENDATION_TABLE_DIR / "recommendation_disparity_by_occupation.csv"
heatmap_path = RECOMMENDATION_FIGURE_DIR / "topk_distribution_heatmap.png"
disparity_plot_path = RECOMMENDATION_FIGURE_DIR / "recommendation_disparity_top_occupations.png"

topk_dist.to_csv(topk_dist_path, index=False)
rec_disparity.to_csv(rec_disparity_path, index=False)
plot_topk_distribution_heatmap(topk_dist, heatmap_path, top_n=30)
plot_recommendation_disparity_bar(rec_disparity, disparity_plot_path, top_n=30)

display(topk_dist.head())
display(rec_disparity.head(20))
print("Saved:", topk_dist_path)
print("Saved:", rec_disparity_path)


## Disparity Analysis

Recommendation disparity is measured as the maximum group top-k rate minus the minimum group top-k rate for each occupation. This is a ranking-output disparity in a controlled simulation, not evidence about real occupations or individual preferences.


## Sensitivity Analysis

The sensitivity grid checks whether disparities are stable across seeds, top-k thresholds, and synthetic history sizes. Stable high-disparity occupations across settings provide stronger evidence than a single run.


In [ ]:
sensitivity_rows = []

for seed in [0, 1, 2, 3, 4]:
    for top_k in [3, 5, 10]:
        for history_size in [5, 10, 20]:
            sens_user_meta, sens_user_emb = simulate_synthetic_users(
                image_embeddings=image_embeddings,
                metadata=sim_metadata,
                group_col=sim_group_col,
                n_users_per_group=100,
                history_size=history_size,
                random_state=seed,
            )
            sens_recommendations = recommend_occupations_for_users(
                user_embeddings=sens_user_emb,
                user_metadata=sens_user_meta,
                text_embeddings=sim_text_embeddings,
                occupation_names=sim_occupation_names,
                top_k=top_k,
            )
            sens_topk_dist = aggregate_topk_distribution(
                sens_recommendations,
                group_col="group",
                all_occupations=sim_occupation_names,
            )
            sens_disparity = compute_recommendation_disparity(sens_topk_dist)
            sens_disparity["seed"] = seed
            sens_disparity["top_k"] = top_k
            sens_disparity["history_size"] = history_size
            sensitivity_rows.append(sens_disparity)

recommendation_sensitivity_results = pd.concat(sensitivity_rows, ignore_index=True)
recommendation_sensitivity_summary = (
    recommendation_sensitivity_results
    .groupby(["occupation", "top_k", "history_size"], as_index=False)
    .agg(
        mean_disparity=("disparity_gap", "mean"),
        std_disparity=("disparity_gap", "std"),
        n_runs=("disparity_gap", "size"),
    )
    .sort_values("mean_disparity", ascending=False)
)

sensitivity_results_path = RECOMMENDATION_TABLE_DIR / "recommendation_sensitivity_results.csv"
sensitivity_summary_path = RECOMMENDATION_TABLE_DIR / "recommendation_sensitivity_summary.csv"
sensitivity_plot_path = RECOMMENDATION_FIGURE_DIR / "recommendation_sensitivity_top_disparities.png"

recommendation_sensitivity_results.to_csv(sensitivity_results_path, index=False)
recommendation_sensitivity_summary.to_csv(sensitivity_summary_path, index=False)
plot_recommendation_sensitivity(recommendation_sensitivity_summary, sensitivity_plot_path, top_n=20)

display(recommendation_sensitivity_summary.head(30))
print("Saved:", sensitivity_results_path)
print("Saved:", sensitivity_summary_path)


## Bootstrap Uncertainty

The bootstrap resamples synthetic users within each demographic group. The resulting interval describes uncertainty over this simulation procedure, not uncertainty about real people or true occupations.


In [ ]:
rec_disparity_bootstrap = bootstrap_recommendation_disparity(
    recommendations,
    group_col="group",
    all_occupations=sim_occupation_names,
    n_bootstrap=200,
    random_state=42,
)

rec_disparity_bootstrap_path = RECOMMENDATION_TABLE_DIR / "recommendation_disparity_bootstrap_ci.csv"
rec_disparity_bootstrap.to_csv(rec_disparity_bootstrap_path, index=False)

display(rec_disparity_bootstrap.head(20))
print("Saved:", rec_disparity_bootstrap_path)


## Report-ready Notes

After running this notebook, use the default disparity table together with the bootstrap CI and sensitivity summary. A careful sentence template is:

"In a controlled retrieval simulation, synthetic profiles constructed from one demographic group produced different top-k occupation recommendation distributions across groups. The largest disparities were observed for [insert occupation after running table], and sensitivity analysis tested whether these patterns persisted across seeds, top-k values, and history sizes."

Do not report placeholders as findings. Replace bracketed text only after inspecting the generated CSV files.


## Limitations and Ethical Caution

This is not a production recommender. It is a controlled simulation designed to test whether CLIP embedding associations can propagate into retrieval outputs. Synthetic users are constructed only from demographic-group image embeddings and do not represent real users. The recommended occupations are CLIP text prompts, not claims about individual occupations, abilities, or preferences.
